In [1]:
from pathlib import Path
import numpy as np
import traceback

from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe


# ============================================================
# 修改成你的实际路径
# ============================================================

INTERMEDIATE_DIR = Path(
    "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models"
)

GOLDEN_DIR = Path(
    "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/golden_pynq"
)

X_GOLDEN = GOLDEN_DIR / "x_int8_golden.npy"
Y_GOLDEN_INT = GOLDEN_DIR / "y_model_out_int_golden.npy"


# ============================================================
# Load golden
# ============================================================

x_all = np.load(str(X_GOLDEN)).astype(np.float32)
y_gold_all = np.load(str(Y_GOLDEN_INT)).astype(np.float32)

x0_flat = x_all[0].reshape(-1)
y_gold = y_gold_all[0].reshape(-1)

print("x0 first 20:", x0_flat[:20])
print("gold y_int:", y_gold)
print("gold pred:", int(np.argmax(y_gold)))


# ============================================================
# Helpers
# ============================================================

def try_get_dtype(model, tensor_name):
    try:
        return model.get_tensor_datatype(tensor_name)
    except Exception:
        return None


def try_get_shape(model, tensor_name):
    try:
        return model.get_tensor_shape(tensor_name)
    except Exception:
        return None


def inspect_model(model_path):
    print("\n" + "=" * 90)
    print("MODEL:", model_path.name)
    print("=" * 90)

    try:
        model = ModelWrapper(str(model_path))
    except Exception as e:
        print("[load failed]", repr(e))
        return

    if len(model.graph.input) == 0 or len(model.graph.output) == 0:
        print("[skip] no graph input/output")
        return

    inp_name = model.graph.input[0].name
    out_name = model.graph.output[0].name

    inp_shape = try_get_shape(model, inp_name)
    out_shape = try_get_shape(model, out_name)

    inp_dtype = try_get_dtype(model, inp_name)
    out_dtype = try_get_dtype(model, out_name)

    print("input :", inp_name)
    print("  shape:", inp_shape)
    print("  dtype:", inp_dtype)

    print("output:", out_name)
    print("  shape:", out_shape)
    print("  dtype:", out_dtype)

    prod = model.find_producer(out_name)
    if prod is None:
        print("  output producer: None")
    else:
        print("  output producer:", prod.name, prod.op_type)
        print("  producer inputs :", list(prod.input))
        print("  producer outputs:", list(prod.output))

    if inp_shape is None:
        print("[skip execute] input shape unknown")
        return

    expected_numel = int(np.prod(inp_shape))

    if expected_numel != x0_flat.size:
        print(
            "[skip execute] input numel mismatch:",
            "model expects", expected_numel,
            "golden has", x0_flat.size,
        )
        return

    x0 = x0_flat.reshape(inp_shape).astype(np.float32)

    print("x0 reshaped:", x0.shape)
    print("x0 first 20:", x0.reshape(-1)[:20])

    try:
        ret = oxe.execute_onnx(model, {inp_name: x0})
        y = ret[out_name].reshape(-1).astype(np.float32)

        print("EXEC OK")
        print("y:", y)
        print("pred:", int(np.argmax(y)))

        if y.shape == y_gold.shape:
            diff = y - y_gold
            print("diff vs Brevitas y_int:")
            print("  max abs :", float(np.max(np.abs(diff))))
            print("  mean abs:", float(np.mean(np.abs(diff))))
            print("  diff    :", diff)
        else:
            print("shape mismatch vs y_gold:", y.shape, y_gold.shape)

    except Exception as e:
        print("[execute failed]", repr(e))
        tb = traceback.format_exc().splitlines()
        print("\n".join(tb[-8:]))


# ============================================================
# Run on all intermediate ONNX files
# ============================================================

onnx_files = sorted(INTERMEDIATE_DIR.glob("*.onnx"))

print("Found ONNX files:")
for p in onnx_files:
    print(" ", p.name)

for p in onnx_files:
    inspect_model(p)

x0 first 20: [ 59.  11. -34. -66. -73. -55. -31. -18. -10.   5.  14.  13.  13.  13.
  12.  11.  12.  10.  10.   9.]
gold y_int: [ 358597. -116274. -164423. -296831. -126583.]
gold pred: 0
Found ONNX files:
  dataflow_parent.onnx
  pre_partition_cleanup.onnx
  step_absorb_mul_pad_mt_patched.onnx
  step_after_convert_to_hw_cleanup.onnx
  step_after_convert_to_hw_cleanup_fmpad.onnx
  step_after_convert_to_hw_cleanup_fmpad_inputfix.onnx
  step_after_streamline_cleanup.onnx
  step_apply_folding_config.onnx
  step_convert_to_hw.onnx
  step_convert_to_hw_after_extra_cleanup.onnx
  step_convert_to_hw_patched.onnx
  step_create_dataflow_partition.onnx
  step_create_dataflow_partition_patched.onnx
  step_create_stitched_ip.onnx
  step_deployment_package.onnx
  step_fix_names_after_specialize.onnx
  step_fix_pad_integer_dtypes.onnx
  step_generate_estimate_reports.onnx
  step_hw_codegen.onnx
  step_hw_ipgen.onnx
  step_make_pynq_driver.onnx
  step_minimize_bit_width.onnx
  step_qonnx_to_finn.onnx

/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor Add_0_param0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(


EXEC OK
y: [-1243.8153    782.186     284.95395  -422.08087   876.2522 ]
pred: 4
diff vs Brevitas y_int:
  max abs : 359840.8125
  mean abs: 213094.625
  diff    : [-359840.8   117056.19  164707.95  296408.9   127459.25]

MODEL: pre_partition_cleanup.onnx
input : global_in
  shape: [1, 1, 140, 1]
  dtype: FLOAT32
output: global_out
  shape: [1, 5, 1, 1]
  dtype: FLOAT32
  output producer: Add_0 Add
  producer inputs : ['Mul_0_out0', 'Add_0_param0']
  producer outputs: ['global_out']
x0 reshaped: (1, 1, 140, 1)
x0 first 20: [ 59.  11. -34. -66. -73. -55. -31. -18. -10.   5.  14.  13.  13.  13.
  12.  11.  12.  10.  10.   9.]
EXEC OK
y: [-1243.8153    782.186     284.95395  -422.08087   876.2522 ]
pred: 4
diff vs Brevitas y_int:
  max abs : 359840.8125
  mean abs: 213094.625
  diff    : [-359840.8   117056.19  164707.95  296408.9   127459.25]

MODEL: step_absorb_mul_pad_mt_patched.onnx
input : global_in
  shape: [1, 1, 140, 1]
  dtype: FLOAT32
output: global_out
  shape: [1, 5, 1, 1]
  d

/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor Add_3_param0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(


EXEC OK
y: [-1243.8153    782.186     284.95395  -422.08087   876.2522 ]
pred: 4
diff vs Brevitas y_int:
  max abs : 359840.8125
  mean abs: 213094.625
  diff    : [-359840.8   117056.19  164707.95  296408.9   127459.25]

MODEL: step_after_convert_to_hw_cleanup.onnx
input : global_in
  shape: [1, 1, 140, 1]
  dtype: FLOAT32
output: global_out
  shape: [1, 5, 1, 1]
  dtype: FLOAT32
  output producer: Add_0 Add
  producer inputs : ['Mul_0_out0', 'Add_0_param0']
  producer outputs: ['global_out']
x0 reshaped: (1, 1, 140, 1)
x0 first 20: [ 59.  11. -34. -66. -73. -55. -31. -18. -10.   5.  14.  13.  13.  13.
  12.  11.  12.  10.  10.   9.]
EXEC OK
y: [-1243.8153    782.186     284.95395  -422.08087   876.2522 ]
pred: 4
diff vs Brevitas y_int:
  max abs : 359840.8125
  mean abs: 213094.625
  diff    : [-359840.8   117056.19  164707.95  296408.9   127459.25]

MODEL: step_after_convert_to_hw_cleanup_fmpad.onnx
input : global_in
  shape: [1, 1, 1, 140]
  dtype: FLOAT32
output: global_out
  shap

In [2]:
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper

MODEL_PATH = Path(
    "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_set_fifo_depths.onnx"
)

model = ModelWrapper(str(MODEL_PATH))

print("Graph outputs:")
for o in model.graph.output:
    name = o.name
    print("\noutput:", name)
    print("  shape:", model.get_tensor_shape(name))
    print("  dtype:", model.get_tensor_datatype(name))

    prod = model.find_producer(name)
    if prod is None:
        print("  producer: None")
    else:
        print("  producer:", prod.name, prod.op_type)
        print("  domain  :", prod.domain)
        print("  inputs  :", list(prod.input))
        print("  outputs :", list(prod.output))

        try:
            from qonnx.custom_op.registry import getCustomOp
            cop = getCustomOp(prod)
            print("  custom op attrs:")
            for k, v in cop.get_nodeattr_types().items():
                try:
                    print("   ", k, "=", cop.get_nodeattr(k))
                except Exception:
                    pass
        except Exception as e:
            print("  custom op attr read failed:", repr(e))

Graph outputs:

output: global_out
  shape: [1, 1, 1, 5]
  dtype: INT32
  producer: StreamingFIFO_rtl_47 StreamingFIFO_rtl
  domain  : finn.custom_op.fpgadataflow.rtl
  inputs  : ['MVAU_rtl_0_out0']
  outputs : ['global_out']
  custom op attrs:
    impl_style = rtl
    backend = fpgadataflow
    preferred_impl_style = 
    code_gen_dir_ipgen = /tmp/finn_dev_slowman/code_gen_ipgen_StreamingFIFO_rtl_47_y_x89d79
    ipgen_path = /tmp/finn_dev_slowman/code_gen_ipgen_StreamingFIFO_rtl_47_y_x89d79
    ip_path = /tmp/finn_dev_slowman/code_gen_ipgen_StreamingFIFO_rtl_47_y_x89d79
    ip_vlnv = 
    exec_mode = 
    cycles_rtlsim = 0
    cycles_estimate = 0
    rtlsim_trace = 
    res_estimate = 
    res_synth = 
    rtlsim_so = 
    slr = -1
    mem_port = 
    partition_id = 0
    device_id = 0
    inFIFODepths = [0]
    outFIFODepths = [0]
    output_hook = 
    io_chrc_in = []
    io_chrc_out = []
    io_chrc_period = 0
    io_chrc_pads_in = []
    io_chrc_pads_out = []
    depth = 2
    fol

In [3]:
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp

MODELS = [
    Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_create_dataflow_partition_patched.onnx"),
    Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_specialize_layers.onnx"),
    Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_hw_codegen.onnx"),
    Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_set_fifo_depths.onnx"),
]

def print_node_attrs(model_path):
    print("\n" + "=" * 100)
    print(model_path.name)
    print("=" * 100)

    model = ModelWrapper(str(model_path))

    print("\nGraph outputs:")
    for o in model.graph.output:
        name = o.name
        print(" output:", name)
        print("  shape:", model.get_tensor_shape(name))
        print("  dtype:", model.get_tensor_datatype(name))
        prod = model.find_producer(name)
        print("  producer:", None if prod is None else (prod.name, prod.op_type, prod.domain))

    print("\nMVAU / MatrixVector nodes:")
    for n in model.graph.node:
        if "MVAU" in n.op_type or "MatrixVector" in n.op_type:
            print("\nnode:", n.name)
            print("op_type:", n.op_type)
            print("domain:", n.domain)
            print("inputs:", list(n.input))
            print("outputs:", list(n.output))

            for t in list(n.input) + list(n.output):
                try:
                    print(" tensor", t, "shape", model.get_tensor_shape(t), "dtype", model.get_tensor_datatype(t))
                except Exception as e:
                    print(" tensor", t, "read failed:", repr(e))

            try:
                cop = getCustomOp(n)
                print(" custom attrs:")
                for k in sorted(cop.get_nodeattr_types().keys()):
                    try:
                        print("  ", k, "=", cop.get_nodeattr(k))
                    except Exception:
                        pass
            except Exception as e:
                print(" custom attr read failed:", repr(e))

for p in MODELS:
    print_node_attrs(p)


step_create_dataflow_partition_patched.onnx

Graph outputs:
 output: MVAU_8_out0
  shape: [1, 1, 1, 5]
  dtype: INT32
  producer: ('MVAU_8', 'MVAU', 'finn.custom_op.fpgadataflow')

MVAU / MatrixVector nodes:

node: MVAU_0
op_type: MVAU
domain: finn.custom_op.fpgadataflow
inputs: ['DuplicateStreams_0_out0', 'MVAU_0_param0', 'MVAU_0_param1']
outputs: ['MVAU_0_out0']
 tensor DuplicateStreams_0_out0 shape [1, 140, 1, 1] dtype INT8
 tensor MVAU_0_param0 shape [1, 16] dtype INT8
 tensor MVAU_0_param1 shape [16, 255] dtype INT32
 tensor MVAU_0_out0 shape [1, 140, 1, 16] dtype INT8
 custom attrs:
   ActVal = -128
   MH = 16
   MW = 1
   PE = 1
   SIMD = 1
   accDataType = INT32
   backend = fpgadataflow
   binaryXnorMode = 0
   code_gen_dir_ipgen = 
   cycles_estimate = 0
   cycles_rtlsim = 0
   device_id = 0
   exec_mode = 
   inFIFODepths = [2]
   inputDataType = INT8
   io_chrc_in = []
   io_chrc_out = []
   io_chrc_pads_in = []
   io_chrc_pads_out = []
   io_chrc_period = 0
   ip_path = 


In [4]:
from pathlib import Path
import numpy as np
import traceback

from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe
from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode

MODEL_PATH = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_hw_ipgen.onnx")
X_GOLDEN = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/golden_pynq/x_int8_golden.npy")

x_all = np.load(str(X_GOLDEN)).astype(np.float32)
x0_flat = x_all[0].reshape(-1)

model = ModelWrapper(str(MODEL_PATH))
model = model.transform(SetExecMode("rtlsim"))

inp_name = model.graph.input[0].name
out_name = model.graph.output[0].name

inp_shape = model.get_tensor_shape(inp_name)
out_shape = model.get_tensor_shape(out_name)

print("input:", inp_name, inp_shape, model.get_tensor_datatype(inp_name))
print("output:", out_name, out_shape, model.get_tensor_datatype(out_name))

x0 = x0_flat.reshape(inp_shape).astype(np.float32)

try:
    ret = oxe.execute_onnx(model, {inp_name: x0})
    y = ret[out_name].reshape(-1)
    print("RTLSIM output:", y)
    print("RTLSIM pred:", int(np.argmax(y)))
except Exception as e:
    print("RTLSIM failed:", repr(e))
    print(traceback.format_exc())

input: global_in [1, 140, 1, 1] INT8
output: global_out [1, 1, 1, 5] INT32
RTLSIM failed: FileNotFoundError(2, 'No such file or directory')
Traceback (most recent call last):
  File "/tmp/ipykernel_173/952977046.py", line 30, in <module>
    ret = oxe.execute_onnx(model, {inp_name: x0})
  File "/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/core/onnx_exec.py", line 179, in execute_onnx
    execute_node(node, execution_context, graph, return_full_exec_context, opset_version)
  File "/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/core/onnx_exec.py", line 53, in execute_node
    ex_cu_node.execute_custom_node(node, context, graph, onnx_opset_version=opset_version)
  File "/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/core/execute_custom_node.py", line 40, in execute_custom_node
    inst.execute_node(context, graph)
  File "/home/slowman/Desktop/project/Thesis/finn/src/finn/custom_op/fpgadataflow/hls/duplicatestreams_hls.py", line 132, in ex

In [5]:
#patch
from pathlib import Path

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp


IN_MODEL = Path(
    "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/"
    "finn_build_pynqz2/intermediate_models/step_create_dataflow_partition_patched.onnx"
)

OUT_MODEL = Path(
    "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/"
    "finn_build_pynqz2/intermediate_models/step_create_dataflow_partition_force_final_hls.onnx"
)


model = ModelWrapper(str(IN_MODEL))

graph_output_names = [o.name for o in model.graph.output]
print("graph outputs:", graph_output_names)

patched = False

for n in model.graph.node:
    if n.op_type != "MVAU":
        continue

    cop = getCustomOp(n)

    try:
        mw = cop.get_nodeattr("MW")
        mh = cop.get_nodeattr("MH")
        no_act = cop.get_nodeattr("noActivation")
    except Exception:
        continue

    is_graph_output_producer = any(out in graph_output_names for out in n.output)
    is_final_classifier = (
        is_graph_output_producer
        or (mw == 4480 and mh == 5 and no_act == 1)
    )

    if is_final_classifier:
        print("\n[patch] Found final classifier MVAU")
        print("  name:", n.name)
        print("  outputs:", list(n.output))
        print("  MW:", mw)
        print("  MH:", mh)
        print("  noActivation:", no_act)
        print("  old preferred_impl_style:", cop.get_nodeattr("preferred_impl_style"))

        cop.set_nodeattr("preferred_impl_style", "hls")

        print("  new preferred_impl_style:", cop.get_nodeattr("preferred_impl_style"))
        patched = True

if not patched:
    raise RuntimeError("Could not find final classifier MVAU to patch.")

model.save(str(OUT_MODEL))
print("\nSaved patched model to:")
print(OUT_MODEL)

graph outputs: ['MVAU_8_out0']

[patch] Found final classifier MVAU
  name: MVAU_8
  outputs: ['MVAU_8_out0']
  MW: 4480
  MH: 5
  noActivation: 1
  old preferred_impl_style: 
  new preferred_impl_style: hls

Saved patched model to:
/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_create_dataflow_partition_force_final_hls.onnx


In [6]:
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp

MODEL = Path(
    "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/"
    "finn_build_pynqz2/intermediate_models/step_create_dataflow_partition_force_final_hls.onnx"
)

model = ModelWrapper(str(MODEL))

for n in model.graph.node:
    if n.op_type == "MVAU":
        cop = getCustomOp(n)
        try:
            if cop.get_nodeattr("MW") == 4480 and cop.get_nodeattr("MH") == 5:
                print("name:", n.name)
                print("op_type:", n.op_type)
                print("preferred_impl_style:", cop.get_nodeattr("preferred_impl_style"))
                print("MW:", cop.get_nodeattr("MW"))
                print("MH:", cop.get_nodeattr("MH"))
                print("noActivation:", cop.get_nodeattr("noActivation"))
                print("outputDataType:", cop.get_nodeattr("outputDataType"))
        except Exception:
            pass

name: MVAU_8
op_type: MVAU
preferred_impl_style: hls
MW: 4480
MH: 5
noActivation: 1
outputDataType: INT32


In [1]:
from qonnx.core.modelwrapper import ModelWrapper
from onnx import helper
import numpy as np

MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_streamline.onnx"

model = ModelWrapper(MODEL_PATH)

def get_attr(node, name, default=None):
    for a in node.attribute:
        if a.name == name:
            return helper.get_attribute_value(a)
    return default

def find_producer(model, tensor_name):
    for n in model.graph.node:
        if tensor_name in n.output:
            return n
    return None

def find_consumers(model, tensor_name):
    return [n for n in model.graph.node if tensor_name in n.input]

print("=" * 100)
print("MultiThreshold report")
print("=" * 100)

for node in model.graph.node:
    if node.op_type == "MultiThreshold":
        print("\n" + "=" * 100)
        print("MT name:", node.name)
        print("inputs :", list(node.input))
        print("outputs:", list(node.output))

        out_scale = get_attr(node, "out_scale")
        out_bias = get_attr(node, "out_bias")
        print("out_scale:", out_scale)
        print("out_bias :", out_bias)

        try:
            print("output dtype:", model.get_tensor_datatype(node.output[0]))
        except Exception as e:
            print("output dtype: <unknown>", e)

        prod = find_producer(model, node.input[0])
        if prod is not None:
            print("producer:", prod.name, prod.op_type)
        else:
            print("producer: <graph input or initializer>")

        consumers = find_consumers(model, node.output[0])
        print("consumers:")
        for c in consumers:
            print("  ", c.name, c.op_type)

MultiThreshold report

MT name: MultiThreshold_0
inputs : ['global_in', 'MultiThreshold_0_param0']
outputs: ['MultiThreshold_0_out0']
out_scale: None
out_bias : -8.0
output dtype: INT4
producer: <graph input or initializer>
consumers:
   Transpose_0 Transpose
   Pad_0 Pad

MT name: MultiThreshold_1
inputs : ['MatMul_0_out0', 'MultiThreshold_1_param0']
outputs: ['MultiThreshold_1_out0']
out_scale: None
out_bias : -2.0
output dtype: INT2
producer: MatMul_0 MatMul
consumers:
   Transpose_2 Transpose

MT name: MultiThreshold_2
inputs : ['MatMul_1_out0', 'MultiThreshold_2_param0']
outputs: ['MultiThreshold_2_out0']
out_scale: 4.0
out_bias : 0.0
output dtype: BIPOLAR
producer: MatMul_1 MatMul
consumers:
   Transpose_3 Transpose

MT name: MultiThreshold_3
inputs : ['MatMul_2_out0', 'MultiThreshold_3_param0']
outputs: ['MultiThreshold_3_out0']
out_scale: None
out_bias : -2.0
output dtype: INT2
producer: MatMul_2 MatMul
consumers:
   Transpose_5 Transpose

MT name: MultiThreshold_4
inputs : ['A